# Dynamic Agents

We can build custom middleware to perform task dynamically. 

For that we need to implement hooks that run at specific points in the agent execution flow. We have below options for that,
1. Node-style hooks - these run at specific execution points, like before_agent, before_model, after_agent, after_model
2. Tool Runtime - these are to update State, Context which are declared.
3. Wrap-style hooks - run around each call, give you control over execution like wrap_model_call, wrap_tool_call

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Prompts

In [2]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest 

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia duit! Tá mé go breá, go raibh maith agat as a fhiafraí. Conas atá tusa?


In [6]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

¡Hola! Estoy bien, gracias por preguntar. ¿Cómo estás tú? ¿En qué puedo ayudarte hoy?


## Tools

In [7]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:
    """Obtain information from the databse using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [8]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [9]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Dynamically call tools based on the runtime context"""
    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass # internal get access to all tools
    else:
        tools = [web_search] # external users only get access to web
        request = request.override(tools=tools)
    
    return handler(request)

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[web_search, sql_query],
    context_schema=UserRole,
    middleware=[dynamic_tool_call]
)

In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don't have access to a specific database that you're referring to. To help you find the number of artists in a database, I need more information:

1. **Which database are you asking about?** For example:
   - A music streaming platform (Spotify, Apple Music, etc.)
   - A specific music database (MusicBrainz, Last.fm, Discogs, etc.)
   - A local database you've created
   - An art platform or museum database
   - Something else?

2. **Do you have access to this database yourself?** If so, I could help you with:
   - Query syntax to count artists
   - How to access the database
   - Interpreting the results

Once you provide more details, I'll be better able to help you!


## Models

In [12]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-5")
standard_model = init_chat_model("claude-haiku-4-5")

@wrap_model_call
def state_based_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    #request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    if message_count > 10:
        model = large_model
    else:
        model = standard_model

    request = request.override(model=model)

    return handler(request)

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [14]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
    ]}
)

print(response["messages"][-1].content)

I appreciate the question, but I should be straightforward with you—I'm Claude, an AI assistant. I don't have a physical form, so I can't actually water plants or be present in an office.

That said, if you're looking for help with office plant care, I'd be happy to:
- Create a watering schedule reminder
- Suggest the best watering practices for specific plants
- Help you set up a system so someone on the team remembers
- Draft a note to leave by the plant as a reminder

Is there something specific about plant care or office organization I can help you with?


In [15]:
print(response["messages"][-1].response_metadata["model_name"])

claude-haiku-4-5-20251001


In [16]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

I need to be honest with you - I should clarify something. I'm actually an AI assistant, not a physical office intern. I can't actually water plants, observe their growth, or rotate pots. 

I apologize for the confusion in my earlier responses. I shouldn't have played along as if I had physically interacted with a real plant.

If you're asking for general advice about plant care, I'm happy to help with that! For pot replacement, most houseplants typically need repotting every 1-2 years when they become rootbound, but it depends on the specific plant species and its growth rate.

Is there something I can actually help you with today?


In [17]:
print(response["messages"][-1].response_metadata["model_name"])

claude-sonnet-4-5-20250929
